# Chapter 16: Query Optimization


## Core Question

How does a query optimizer use statistics to choose an estimated low-cost plan without
changing the meaning of the SQL query?

Use this guide with the executable SQL lab cell.

Chapter 15 and Chapter 16 share one class meeting. The classroom core is limited to
result equivalence, basic selectivity, catalog statistics, `ANALYZE`, and plan comparison.
Formal rewrite systems, detailed join enumeration, skew analysis, and optimizer
implementation are extensions.


## Connection to Chapter 15

Chapter 15 distinguished a logical expression from a physical plan and introduced scans,
index searches, and join order. This chapter explains how statistics help an optimizer
estimate alternatives. A cost estimate is a decision input, not a measurement of actual
runtime.


## Prerequisites

- Confirm whether two queries return the same rows and duplicate counts on sample data.
- Read selections and inner or outer joins.
- Identify scans, index searches, and access order in a Chapter 15 query plan.


## Learning Objectives

After completing the classroom core, you should be able to:

1. Explain why result equivalence must be checked before performance comparison.
2. Describe how row counts, distinct values, and indexes support cost estimation.
3. Calculate a basic equality-selectivity estimate and state its assumptions.
4. Run `ANALYZE` and interpret bounded evidence from `EXPLAIN QUERY PLAN`.
5. Compare equivalent query results and plans without claiming more than the evidence
   supports.


## Teaching Summary

| Classroom topic | Worked evidence | Evidence to retain |
|---|---|---|
| Equivalence before optimization | Base and rewritten query | Bidirectional difference check |
| Statistics and selectivity | Event-type distribution | Estimated and actual row counts |
| `ANALYZE` and plan choice | SQLite statistics and plans | Statistics snapshot and plan |
| Evidence limits | Estimated versus actual behavior | One bounded conclusion |


## 1. What an Optimizer Does

An optimizer may consider logically equivalent expressions, physical access paths, and
join orders. It uses available statistics to estimate their cost and selects one plan.
The estimated lowest-cost plan is not guaranteed to be the fastest in every execution,
because statistics and modeling assumptions may be incomplete or stale.

### Worked Example

For a query joining Student, Enrollment, and Course, the optimizer may consider different
join orders and choose a scan or search for each input. These choices are acceptable only
if the resulting query preserves join predicates, duplicate behavior, `NULL` behavior,
and required output columns.

### Predict Before Checking

Before comparing plans for two SQL statements, list the properties that must remain the
same. Include the result columns, row values, duplicate counts, join types, and filters.

### Interpretation

Formatting two statements differently does not establish a useful optimization. Equal
results on one data set are necessary evidence, but they do not prove equivalence for all
legal future data.


## 2. Result Equivalence as a Guardrail

Two relational expressions are equivalent when they produce the same result for every
legal database instance. SQL usually preserves duplicates, so duplicate counts also
matter. Constraints, `NULL`, and outer joins can make an apparently simple rewrite
incorrect.

### Lab Activity

Compare the lab's `BASE QUERY` and `PUSHDOWN QUERY`:

1. Predict whether they should return the same result.
2. Run both queries.
3. Use bidirectional `EXCEPT` checks and row counts.
4. Compare the two query plans only after the result checks pass.

### Expected Interpretation

In the supplied data, both difference counts are zero and the row counts match. SQLite
may flatten the subqueries and show the same physical plan. This supports equality for
the tested data and demonstrates the observed optimizer behavior. It does not prove that
every arbitrary subquery can be removed safely.

SQLite's `EXCEPT` removes duplicate result rows. Therefore, bidirectional `EXCEPT` plus
one total row count is not a general proof that duplicate multiplicities match. When
duplicates are possible, compare grouped `COUNT(*)` values over all result columns or use
another verified comparison that preserves multiplicity.


## 3. Statistics and Selectivity

Optimizers commonly use relation row or page counts, tuple width, distinct-value counts,
index properties, and value-distribution information. Statistics may be sampled or
periodically updated, so they can become stale.

Without value-frequency information, a simple equality estimate may use a uniform
distribution assumption:

```text
estimated rows = total rows / number of distinct values
selectivity = estimated rows / total rows
```

This is an approximation, not a law about the data.

### Worked Example

An Event table has 10,000 rows and two distinct event types. A uniform estimate predicts
5,000 rows for either type. The actual counts are:

```text
COMMON: 9,900
RARE:      100
```

The estimate is wrong in opposite directions for the two values. The actual selectivity
is 99% for `COMMON` and 1% for `RARE`.

### Predict and Check

Before examining a plan, predict which value is more likely to benefit from a secondary
index lookup. Then state why the selectivity percentage still does not guarantee the
chosen plan.

### Interpretation

`RARE` is more likely to benefit because it returns fewer rows. The final choice may also
depend on whether the index covers the query, row placement, cache state, and the
optimizer's available statistics.


## 4. `ANALYZE` and Plan Evidence

SQLite `ANALYZE` records planner statistics. In this lab, selected values can be viewed
through `sqlite_stat1`:

```sql
ANALYZE;
SELECT tbl, idx, stat
FROM sqlite_stat1
ORDER BY tbl, idx;
```

SQLite's statistics format is product-specific and more compact than a full textbook
catalog. Do not treat one `stat` string as a standard format for every DBMS.

### Lab Activity

1. Record the DBMS version, schema, indexes, and row counts.
2. Run `ANALYZE` and retain the relevant statistics rows.
3. Run the base and rewritten queries.
4. Confirm equal results.
5. Explain the access order, `SCAN` or `SEARCH`, and index names shown by each plan.
6. Identify one piece of actual execution information that the compact plan omits.

### Check Criteria

A complete answer separates observed facts from inference. Examples of omitted data
include actual rows per operator, elapsed time, buffer reads, and memory use.


## 5. A Practical Optimization Record

For a defensible comparison, retain this sequence:

1. SQL, schema, indexes, row counts, and DBMS version.
2. Evidence that the query results are equal.
3. Statistics state, including whether `ANALYZE` was run.
4. Both query plans.
5. A conclusion limited to the tested data and environment.

An optimizer estimate can guide a plan choice, but it is not an actual measurement. If
elapsed time matters, repeat the measurement under controlled conditions and keep the
plans and results with the timings.


## Extensions: Rewrite Rules and Optimizer Internals

The following topics remain available for after-class reading but are not part of the
Exam 3 classroom core:

- selection and projection pushdown rules;
- detailed join reordering and Cartesian-product avoidance;
- outer-join rewrite counterexamples;
- histogram and frequent-value handling for skew;
- complete cost formulas and dynamic-programming plan enumeration;
- optimizer implementation details.

One important warning is still required: moving a right-side condition between `ON` and
`WHERE` in a left outer join can change unmatched rows because `NULL` comparisons become
`UNKNOWN`. Never apply an inner-join rewrite rule to an outer join without checking its
semantics.

### Extension Counterexample

```sql
-- Filter after the left join
FROM department AS d
LEFT JOIN student AS s ON s.dept_id = d.dept_id
WHERE s.student_id < 3

-- Filter as part of the join condition
FROM department AS d
LEFT JOIN student AS s
  ON s.dept_id = d.dept_id AND s.student_id < 3
```

A department with no student is removed by the first form and retained with a `NULL`
student by the second. This is a semantic difference, not a performance detail.


## Common Errors

1. Declaring universal equivalence from one sample result.
2. Comparing plans before checking result equality.
3. Treating estimated rows as actual rows.
4. Assuming statistics remain current after major data changes.
5. Treating low selectivity as a guarantee that an index will be used.
6. Claiming that SQL formatting changes the physical algorithm.
7. Applying an inner-join rewrite to an outer join without a counterexample check.


## Discussion and Individual Evidence

Compare a base query and a proposed rewrite. First support or reject result equivalence.
Then inspect statistics and plans. Finish with one conclusion the evidence supports and
one conclusion it does not support.

Retain the original and rewritten SQL, bidirectional difference check, statistics
snapshot, both plans, actual counts, and one unsafe-rewrite counterexample.


## Chapter Summary

Query optimization begins with meaning, then uses statistics to estimate alternatives.
The classroom core is to verify equal results, calculate simple selectivity, inspect
statistics, and interpret plans conservatively. Formal rewrite systems, skew handling,
and optimizer implementation remain extensions. Chapter 17 moves from one query plan to
transactions and concurrent schedules.


## After-Class Continuation

Create a four-part record for one query: equivalence, statistics, plan, and limitations.
Read the detailed rewrite and optimizer topics as extensions; they are not required
derivations for the selected Chapter 16 assessment.


## Executable Notebook Lab

Predict before running each cell, then compare the output with your explanation.


In [1]:
import sqlite3

print(f"Python {__import__('sys').version.split()[0]}; SQLite {sqlite3.sqlite_version}")
connection = sqlite3.connect(":memory:", isolation_level=None)
connection.execute("PRAGMA foreign_keys = ON")


def run_sql_script(connection, script, max_rows=20):
    """Execute a SQLite script and display result-producing statements."""
    buffer = ""
    for raw_line in script.splitlines():
        stripped = raw_line.strip()
        if stripped.startswith(".print"):
            message = stripped[len(".print"):].strip().strip("\"'")
            print(f"\n{message}")
            continue
        buffer += raw_line + "\n"
        if not sqlite3.complete_statement(buffer):
            continue
        statement = buffer.strip()
        buffer = ""
        if not statement:
            continue
        cursor = connection.execute(statement)
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchmany(max_rows + 1)
            print(" | ".join(columns))
            for row in rows[:max_rows]:
                print(" | ".join("NULL" if value is None else str(value) for value in row))
            if len(rows) > max_rows:
                print(f"... additional rows omitted after {max_rows}")
    remaining = "\n".join(
        line for line in buffer.splitlines() if not line.strip().startswith("--")
    ).strip()
    if remaining:
        raise ValueError("The embedded SQL ends with an incomplete statement.")


Python 3.12.13; SQLite 3.53.1


### Query-optimization lab


In [2]:
SQL_1 = """PRAGMA foreign_keys = ON;
PRAGMA automatic_index = OFF;

DROP INDEX IF EXISTS ch16_idx_student_dept;
DROP INDEX IF EXISTS ch16_idx_course_credits;
DROP INDEX IF EXISTS ch16_idx_enrollment_course;
DROP INDEX IF EXISTS ch16_idx_event_type;
DROP TABLE IF EXISTS ch16_enrollment;
DROP TABLE IF EXISTS ch16_student;
DROP TABLE IF EXISTS ch16_course;
DROP TABLE IF EXISTS ch16_department;
DROP TABLE IF EXISTS ch16_event;

CREATE TABLE ch16_department (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE
);

CREATE TABLE ch16_student (
    student_id INTEGER PRIMARY KEY,
    dept_id    INTEGER NOT NULL,
    name       TEXT NOT NULL,
    FOREIGN KEY (dept_id) REFERENCES ch16_department(dept_id)
);

CREATE TABLE ch16_course (
    course_id INTEGER PRIMARY KEY,
    title     TEXT NOT NULL,
    credits   INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 5)
);

CREATE TABLE ch16_enrollment (
    student_id INTEGER NOT NULL,
    course_id  INTEGER NOT NULL,
    grade      TEXT,
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES ch16_student(student_id),
    FOREIGN KEY (course_id) REFERENCES ch16_course(course_id)
);

WITH RECURSIVE seq(n) AS (
    VALUES (1)
    UNION ALL
    SELECT n + 1 FROM seq WHERE n < 101
)
INSERT INTO ch16_department(dept_id, dept_name)
SELECT n, printf('Department %03d', n)
FROM seq;

WITH RECURSIVE seq(n) AS (
    VALUES (1)
    UNION ALL
    SELECT n + 1 FROM seq WHERE n < 10000
)
INSERT INTO ch16_student(student_id, dept_id, name)
SELECT n, ((n - 1) % 100) + 1, printf('Student %05d', n)
FROM seq;

WITH RECURSIVE seq(n) AS (
    VALUES (1)
    UNION ALL
    SELECT n + 1 FROM seq WHERE n < 500
)
INSERT INTO ch16_course(course_id, title, credits)
SELECT n, printf('Course %03d', n), ((n - 1) % 5) + 1
FROM seq;

WITH RECURSIVE students(n) AS (
    VALUES (1)
    UNION ALL
    SELECT n + 1 FROM students WHERE n < 10000
), five(k) AS (VALUES (1), (2), (3), (4), (5))
INSERT INTO ch16_enrollment(student_id, course_id, grade)
SELECT n,
       ((n * 17 + k * 97) % 500) + 1,
       CASE k WHEN 1 THEN 'A' WHEN 2 THEN 'B+' WHEN 3 THEN 'B'
              WHEN 4 THEN 'A-' ELSE 'C+' END
FROM students CROSS JOIN five;

CREATE INDEX ch16_idx_student_dept ON ch16_student(dept_id);
CREATE INDEX ch16_idx_course_credits ON ch16_course(credits);
CREATE INDEX ch16_idx_enrollment_course ON ch16_enrollment(course_id);

-- A deliberately skewed column for statistics reasoning.
CREATE TABLE ch16_event (
    event_id   INTEGER PRIMARY KEY,
    event_type TEXT NOT NULL,
    amount     NUMERIC NOT NULL
);

WITH RECURSIVE seq(n) AS (
    VALUES (1)
    UNION ALL
    SELECT n + 1 FROM seq WHERE n < 10000
)
INSERT INTO ch16_event(event_id, event_type, amount)
SELECT n,
       CASE WHEN n <= 9900 THEN 'COMMON' ELSE 'RARE' END,
       round(10 + (n % 1000) / 10.0, 2)
FROM seq;

CREATE INDEX ch16_idx_event_type ON ch16_event(event_type);
ANALYZE;

SELECT COUNT(*) AS students FROM ch16_student;
SELECT COUNT(*) AS enrollments FROM ch16_enrollment;

-- === BASE QUERY ===
EXPLAIN QUERY PLAN
SELECT s.student_id, c.course_id
FROM ch16_student AS s
JOIN ch16_enrollment AS e ON e.student_id = s.student_id
JOIN ch16_course AS c ON c.course_id = e.course_id
WHERE s.dept_id = 42 AND c.credits = 5;

-- === PUSHDOWN QUERY ===
EXPLAIN QUERY PLAN
SELECT s.student_id, c.course_id
FROM (SELECT student_id FROM ch16_student WHERE dept_id = 42) AS s
JOIN ch16_enrollment AS e ON e.student_id = s.student_id
JOIN (SELECT course_id FROM ch16_course WHERE credits = 5) AS c
  ON c.course_id = e.course_id;

-- Compare equivalent results in both directions.
WITH base AS (
    SELECT s.student_id, c.course_id
    FROM ch16_student AS s
    JOIN ch16_enrollment AS e ON e.student_id = s.student_id
    JOIN ch16_course AS c ON c.course_id = e.course_id
    WHERE s.dept_id = 42 AND c.credits = 5
), pushed AS (
    SELECT s.student_id, c.course_id
    FROM (SELECT student_id FROM ch16_student WHERE dept_id = 42) AS s
    JOIN ch16_enrollment AS e ON e.student_id = s.student_id
    JOIN (SELECT course_id FROM ch16_course WHERE credits = 5) AS c
      ON c.course_id = e.course_id
)
SELECT (SELECT COUNT(*) FROM base) AS base_rows,
       (SELECT COUNT(*) FROM pushed) AS pushed_rows,
       (SELECT COUNT(*) FROM (SELECT * FROM base EXCEPT SELECT * FROM pushed))
         AS base_minus_pushed,
       (SELECT COUNT(*) FROM (SELECT * FROM pushed EXCEPT SELECT * FROM base))
         AS pushed_minus_base;

-- === OUTER JOIN COUNTEREXAMPLE ===
-- WHERE rejects the NULL-extended row for Department 101.
SELECT d.dept_id, s.student_id
FROM ch16_department AS d
LEFT JOIN ch16_student AS s ON s.dept_id = d.dept_id
WHERE s.student_id < 3
ORDER BY d.dept_id, s.student_id;

-- ON retains Department 101 with a NULL student_id.
SELECT d.dept_id, s.student_id
FROM ch16_department AS d
LEFT JOIN ch16_student AS s
  ON s.dept_id = d.dept_id AND s.student_id < 3
WHERE d.dept_id IN (1, 2, 101)
ORDER BY d.dept_id, s.student_id;

-- === STATISTICS AND SKEW ===
SELECT tbl, idx, stat
FROM sqlite_stat1
WHERE tbl IN ('ch16_student', 'ch16_course', 'ch16_enrollment', 'ch16_event')
ORDER BY tbl, idx;

SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT event_type) AS distinct_types,
       COUNT(*) / COUNT(DISTINCT event_type) AS uniform_estimate_per_type
FROM ch16_event;

SELECT event_type, COUNT(*) AS actual_rows,
       round(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM ch16_event), 1)
         AS actual_percent
FROM ch16_event
GROUP BY event_type
ORDER BY event_type;

EXPLAIN QUERY PLAN
SELECT event_id, event_type, amount
FROM ch16_event
WHERE event_type = 'RARE';

EXPLAIN QUERY PLAN
SELECT event_id, event_type, amount
FROM ch16_event
WHERE event_type = 'COMMON';
"""


In [3]:
run_sql_script(connection, SQL_1)


students
10000
enrollments
50000
id | parent | notused | detail
4 | 0 | 77 | SEARCH s USING COVERING INDEX ch16_idx_student_dept (dept_id=?)
8 | 0 | 47 | SEARCH e USING COVERING INDEX sqlite_autoindex_ch16_enrollment_1 (student_id=?)
12 | 0 | 35 | SEARCH c USING COVERING INDEX ch16_idx_course_credits (credits=? AND rowid=?)
id | parent | notused | detail
4 | 0 | 77 | SEARCH ch16_student USING COVERING INDEX ch16_idx_student_dept (dept_id=?)
8 | 0 | 47 | SEARCH e USING COVERING INDEX sqlite_autoindex_ch16_enrollment_1 (student_id=?)
12 | 0 | 35 | SEARCH ch16_course USING COVERING INDEX ch16_idx_course_credits (credits=? AND rowid=?)
base_rows | pushed_rows | base_minus_pushed | pushed_minus_base
100 | 100 | 0 | 0
dept_id | student_id
1 | 1
2 | 2
dept_id | student_id
1 | 1
2 | 2
101 | NULL
tbl | idx | stat
ch16_course | ch16_idx_course_credits | 500 100
ch16_enrollment | ch16_idx_enrollment_course | 50000 100
ch16_enrollment | sqlite_autoindex_ch16_enrollment_1 | 50000 5 1
ch16_event | c

### Reproducibility Check


In [4]:
expected = {"ch16_department": 101, "ch16_student": 10000, "ch16_course": 500, "ch16_enrollment": 50000, "ch16_event": 10000}
for table, count in expected.items():
    assert connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0] == count
assert connection.execute("SELECT COUNT(*) FROM ch16_event WHERE event_type='RARE'").fetchone()[0] == 100
assert connection.execute("PRAGMA foreign_key_check").fetchall() == []
print("Notebook checks passed.")


Notebook checks passed.


In [5]:
connection.close()
print("In-memory database closed.")


In-memory database closed.
